# BI Jobs & Question Batch Worker Architecture

- **Recipe File**: `jobs/bi_materialization.py`
- **Jobs**:
  1. `BI_METRIC_EXTRACTION_JOB` (DAG 파이프라인): 재무 지표 추출
  2. `BI_MATERIALIZATION_JOB` (Worker Job): 기업별 대시보드 스냅샷 생성 코디네이터
  3. `BI_QUESTION_JOB` (Worker Job): 16개 배치 & 멀티스레드 병렬 질문 처리기

## 1. 개요 및 구조
BI 시스템은 대규모 기업의 18개 재무 지표 시계열 데이터를 고속으로 추출하기 위해, 전체 질문을 16개 배치 단위로 청킹하고 KEDA 오토스케일링을 통해 병렬 Pod에서 ThreadPoolExecutor로 분산 실행합니다.

In [ ]:
import sys
from pathlib import Path
import json

# Find repository root by walking up from cwd to find jobs/__init__.py
current = Path.cwd()
PROJECT_ROOT = None
for parent in [current] + list(current.parents):
    if (parent / "jobs" / "__init__.py").exists():
        PROJECT_ROOT = parent
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find repository root containing jobs/__init__.py")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jobs.bi_materialization import (
    BI_METRIC_EXTRACTION_JOB,
    BI_MATERIALIZATION_JOB,
    BI_QUESTION_JOB,
)

print(f"📌 [DAG] {BI_METRIC_EXTRACTION_JOB.name} ({BI_METRIC_EXTRACTION_JOB.job_id})")
print(f"📌 [Worker] {BI_MATERIALIZATION_JOB.name} ({BI_MATERIALIZATION_JOB.job_id})")
print(f"   - Entrypoint: {BI_MATERIALIZATION_JOB.worker_entrypoint}")
print(f"   - Queue: {BI_MATERIALIZATION_JOB.queue_name}")
print(f"📌 [Worker] {BI_QUESTION_JOB.name} ({BI_QUESTION_JOB.job_id})")
print(f"   - Entrypoint: {BI_QUESTION_JOB.worker_entrypoint}")
print(f"   - Queue: {BI_QUESTION_JOB.queue_name}")